### LangChain 쓰지 않고 전처리
#### 1. Document Loading


In [ ]:
%pip install python-docx

In [ ]:
from docx import Document

document = Document('./test.docx')
print(document)
dir(document)
full_text = ''
for index, paragraph in enumerate(document.paragraphs):
    full_text += paragraph.text 
    # if(index < 10):
    #     print(f'paragraph={paragraph.text}')


In [ ]:
full_text

### 2. Document Parsing 

In [ ]:
%pip install tiktoken

In [25]:
import tiktoken

def split_text(full_text, chunk_size):
    encoder= tiktoken.encoding_for_model("gpt-4o")
    total_encoding = encoder.encode(full_text)
    total_token_count = len(total_encoding)
    text_list = []
    for i in range(0, total_token_count, chunk_size):
        chunk = total_encoding[i: i+chunk_size]
        decoded= encoder.decode(chunk)
        text_list.append(decoded)
    return text_list

In [26]:
chunk_list = split_text(full_text, 1500)

In [ ]:
chunk_list

### 3. Embedding 

In [ ]:
%pip install chromadb

In [29]:
import chromadb 

chroma_client = chromadb.Client()

In [33]:
collection_name = 'tax_collection'
tax_collection = chroma_client.create_collection(collection_name)

In [47]:
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction 
ollama_embedding = OllamaEmbeddingFunction(model_name="nomic-embed-text:latest")

In [48]:
chroma_client.delete_collection("tax_collection")
tax_collection = chroma_client.get_or_create_collection(collection_name, 
                                                        embedding_function=ollama_embedding)

In [49]:
id_list = []
for index in range(len(chunk_list)):
    id_list.append(f'{index}')
print(len(id_list))
print(len(chunk_list))

107
107


#### 4. Similarity Search

In [50]:
tax_collection.add(documents=chunk_list, ids=id_list)

In [51]:
query ='연봉 5천만원인 직장인의 소득세는 얼마인가요?'
retrieved_doc = tax_collection.query(query_texts=query)

In [ ]:
retrieved_doc['documents'][0]

### 5. LLM 

In [ ]:
%pip install openai

In [ ]:
from openai import OpenAI
client = OpenAI()

response = client.chat.completions.create(
    model="chatgpt-4o-latest",
    messages = [
        {"role":"system", "content" : f"당신은 한국의 소득세 전문가 입니다. 아래의 내요을 참고해서 사용자의 질문에 답변해주세요{retrieved_doc['documents'][0]}"}, 
        {"role" : "user", "content" : query},
    ],
) 

In [ ]:
import ollama

response = ollama.chat(
    model="gemma4:latest",
    messages=[
        {
            "role": "system",
            "content": f"""
당신은 한국의 소득세 전문가 입니다.
아래 내용을 참고해서 사용자의 질문에 답변해주세요.

{retrieved_doc['documents'][0]}
"""
        },
        {
            "role": "user",
            "content": query
        }
    ]
)

print(response['message']['content'])